# Local Whisper no Google Colab

Gera legendas `.srt` em pt-BR a partir de vídeos no Google Drive usando [faster-whisper](https://github.com/SYSTRAN/faster-whisper) com GPU e inferência batched.

**Antes de rodar:** `Ambiente de execução` → `Alterar tipo de ambiente de execução` → **T4 GPU**.

Fluxo:
1. Verificar GPU
2. Instalar dependências
3. Montar o Google Drive
4. Configurar pasta de entrada, modelo e `BATCH_SIZE`
5. Carregar o modelo via `BatchedInferencePipeline`
6. Transcrever e salvar `.srt` ao lado de cada vídeo

> O `BatchedInferencePipeline` processa várias janelas de VAD em paralelo na mesma GPU — ganho típico de **2–4×** sobre o modo sequencial. Em troca, `condition_on_previous_text` é forçado a `False` (os batches são independentes), o que raramente é perceptível na saída.

## 1. Verificar GPU

Se `nvidia-smi` falhar, o ambiente não está com GPU — troque o runtime antes de continuar.

In [ ]:
!nvidia-smi

## 2. Instalar dependências

`faster-whisper` traz `ctranslate2` com CUDA/cuDNN embutidos. `av<14` reproduz a pin do projeto local.

In [ ]:
!pip install -q "faster-whisper>=1.2.1" "av<14"

## 3. Montar o Google Drive

Os vídeos devem estar em alguma pasta do seu Drive. Cada legenda será salva com o mesmo nome do vídeo (`video.mp4` → `video.srt`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configuração

- **INPUT_PATH**: pasta ou arquivo de vídeo dentro do Drive.
- **OUTPUT_PATH**: pasta de saída (vazio = salvar ao lado do vídeo).
- **MODEL_SIZE**: `large-v3` aproveita o T4 e dá o melhor resultado; use `small` para igualar o setup local.
- **BATCH_SIZE**: janelas de VAD processadas em paralelo na GPU. `8`–`16` costuma ser o sweet spot no T4 com `large-v3`; baixe se estourar VRAM, suba em GPUs maiores.
- **FORCE**: `True` reprocessa mesmo que já exista uma legenda.

In [ ]:
INPUT_PATH = "/content/drive/MyDrive/videos"  # @param {type:"string"}
OUTPUT_PATH = ""  # @param {type:"string"}
MODEL_SIZE = "large-v3"  # @param ["tiny", "base", "small", "medium", "large-v3"]
LANGUAGE = "pt"  # @param {type:"string"}
BEAM_SIZE = 5  # @param {type:"integer"}
BATCH_SIZE = 8  # @param {type:"integer"}
FORCE = False  # @param {type:"boolean"}

## 5. Funções auxiliares

Adaptação do pacote `local_whisper` (`files.py`, `srt.py`, `transcriber.py`) para rodar em um único notebook.

In [ ]:
from __future__ import annotations

import time
from pathlib import Path

from tqdm.auto import tqdm

VIDEO_EXTENSIONS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".flv", ".wmv", ".m4v", ".ts"}

SUBTITLE_EXTENSIONS = {
    ".srt", ".vtt", ".ass", ".ssa", ".sub", ".sbv",
    ".ttml", ".dfxp", ".smi", ".stl", ".usf", ".idx",
}


def collect_videos(path: str) -> list[Path]:
    p = Path(path)
    if p.is_dir():
        return sorted(f for f in p.iterdir() if f.suffix.lower() in VIDEO_EXTENSIONS)
    if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS:
        return [p]
    return []


def existing_subtitle(video: Path, extra_dir: Path | None = None) -> Path | None:
    dirs = [video.parent]
    if extra_dir and extra_dir != video.parent:
        dirs.append(extra_dir)
    for d in dirs:
        for ext in SUBTITLE_EXTENSIONS:
            candidate = d / (video.stem + ext)
            if candidate.exists():
                return candidate
    return None


def format_timestamp(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def segments_to_srt(segments: list) -> str:
    lines: list[str] = []
    for i, seg in enumerate(segments, start=1):
        lines.append(str(i))
        lines.append(f"{format_timestamp(seg.start)} --> {format_timestamp(seg.end)}")
        lines.append(seg.text.strip())
        lines.append("")
    return "\n".join(lines)


def transcribe_file(pipeline, video: Path, output_dir: Path | None,
                    index: int = 0, total: int = 0,
                    language: str = "pt", beam_size: int = 5,
                    batch_size: int = 8) -> Path:
    dest = (output_dir or video.parent) / (video.stem + ".srt")
    batch = f" ({index}/{total})" if total else ""
    print(f"  → {video.name}{batch}")

    start = time.perf_counter()
    segments_gen, info = pipeline.transcribe(
        str(video),
        language=language,
        beam_size=beam_size,
        batch_size=batch_size,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500),
    )

    duration = info.duration
    collected = []
    bar = tqdm(
        total=int(duration),
        unit="s",
        bar_format="    {l_bar}{bar}| {n_fmt}/{total_fmt}s [{elapsed}<{remaining}]",
        leave=False,
    )
    prev_end = 0.0
    for seg in segments_gen:
        collected.append(seg)
        delta = seg.end - prev_end
        if delta > 0:
            bar.update(int(delta))
            prev_end = seg.end
    bar.update(bar.total - bar.n)
    bar.close()

    elapsed = time.perf_counter() - start

    if not collected:
        print(f"    ⚠ nenhuma fala detectada ({elapsed:.1f}s)")
        return dest

    collected.sort(key=lambda s: s.start)
    dest.write_text(segments_to_srt(collected), encoding="utf-8")
    print(f"    ✓ {len(collected)} segmentos — {elapsed:.1f}s → {dest.name}")
    return dest

## 6. Carregar o modelo

Usa `float16` na GPU (rápido e enxuto na VRAM do T4) ou `int8` como fallback de CPU. O modelo é então envolvido pelo `BatchedInferencePipeline`, que faz a inferência em paralelo.

In [ ]:
import ctranslate2
from faster_whisper import BatchedInferencePipeline, WhisperModel

if ctranslate2.get_cuda_device_count() > 0:
    device, compute_type = "cuda", "float16"
else:
    device, compute_type = "cpu", "int8"
    print("[!] GPU não detectada — rodando em CPU (vai ser lento). Troque o runtime para T4.")

model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)
pipeline = BatchedInferencePipeline(model=model)
print(f"[*] Modelo: {MODEL_SIZE} | Dispositivo: {device} ({compute_type}) | batch_size={BATCH_SIZE}")

## 7. Transcrever

Pula vídeos que já têm legenda (a menos que `FORCE=True`). Erros em um arquivo não interrompem o lote.

In [ ]:
videos = collect_videos(INPUT_PATH)
if not videos:
    raise SystemExit(f"Nenhum vídeo encontrado em {INPUT_PATH}")

output_dir = Path(OUTPUT_PATH) if OUTPUT_PATH else None
if output_dir:
    output_dir.mkdir(parents=True, exist_ok=True)

if not FORCE:
    filtered = []
    for v in videos:
        sub = existing_subtitle(v, output_dir)
        if sub:
            print(f"[=] pulando {v.name} — já tem legenda ({sub.name})")
        else:
            filtered.append(v)
    videos = filtered

if not videos:
    print("\n[✓] Nada a fazer — todos os vídeos já têm legenda")
else:
    print(f"[*] {len(videos)} vídeo(s) para transcrever\n")
    for i, video in enumerate(videos, start=1):
        try:
            transcribe_file(pipeline, video, output_dir,
                            index=i, total=len(videos),
                            language=LANGUAGE, beam_size=BEAM_SIZE,
                            batch_size=BATCH_SIZE)
        except Exception as e:
            print(f"    ✗ erro em {video.name}: {e}")
    print("\n[✓] Concluído")